# Emotion-Aware Multi-Voice Generative Audiobook System
## Complete Pipeline — Steps 1 through 7

**Project:** Emotion-Aware Multi-Voice Generative Audiobook System using LLMs and Multimodal Generative AI


| Step | Component | Technology |
|------|-----------|------------|
| 1 | LLM-Based Text Analysis | Gemini 2.5 Flash (free API) |
| 2 | Voice Mapping & Profiles | Custom VoiceMapper + RAG-style retrieval |
| 3 | TTS Synthesis | Parler-TTS Expresso (emotional) + edge-tts (fast) |
| 4 | SFX / Background Ambience | Keyword-matched free SFX library |
| 5 | Audio Stitching & Post-processing | pydub |
| 6 | Human-in-the-Loop Validation | Gradio interactive UI |
| 7 | Evaluation Metrics | MOS, WER (Whisper), Emotion F1, RTF |

**Pre-requisite:** Get a free Gemini API key from [aistudio.google.com](https://aistudio.google.com/app/apikey)

---
## Step 0 — Setup & Installation

This installs everything needed for the full pipeline. Takes about 2-3 minutes.

In [2]:
# ================================================================
# INSTALLATION — Run this cell first, then restart runtime if asked
# ================================================================
!pip install --upgrade pip setuptools
!pip install -q google-generativeai edge-tts pydub nltk soundfile
!pip install -q openai-whisper jiwer gradio
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q transformers==4.30.2 # Pin transformers version for diffusers compatibility
!pip install -q diffusers accelerate scipy # Already added for AudioLDM fix, ensure it's here
!apt-get -qq install ffmpeg

print("\n" + "="*50)
print("Installation complete.")
print("If you see version conflict warnings above, they")
print("are usually harmless. Proceed to the next cell.")
print("="*50)

INSTALL_FIX = '''
# Add this to your installation cell (Step 0):
!pip install -q diffusers accelerate scipy
# This is needed for AudioLDM (SFX generation)
'''

if __name__ == "__main__":
    print("=" * 60)
    print("  AUDIOBOOK PIPELINE FIXES")
    print("=" * 60)
    print()
    print("This file contains 3 fixes. Apply them by replacing")
    print("the corresponding cells in your Colab notebook:")
    print()
    print("FIX 1 (SFX): Replace Step 4 cell with FIX1_CELL_A + FIX1_CELL_B")
    print("  - Uses AudioLDM diffusion model for realistic SFX")
    print("  - Text prompts like 'a door creaking open slowly'")
    print("  - Falls back to improved procedural if AudioLDM fails")
    print()
    print("FIX 2 (edge-tts): Replace Step 3A cell with FIX2_CELL")
    print("  - Pads very short text (< 2 words)")
    print("  - Clamps prosody to safe ranges (±40% rate, ±30Hz pitch)")
    print("  - 3-attempt retry with fallback to neutral prosody")
    print("  - Skips gracefully instead of crashing")
    print()
    print("FIX 3 (Gradio): Replace Step 6 cell with FIX3_CELL")
    print("  - Uses loop.run_until_complete() instead of asyncio.run()")
    print("  - Proper pandas DataFrame for the segment table")
    print("  - Prosody clamping in re-synthesis too")
    print("  - Cleaner UI layout")
    print()
    print("ALSO: Add to your install cell:")
    print("  !pip install -q diffusers accelerate scipy")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
descript-audiotools 0.7.4 requires protobuf!=4.24.0,<5.0.0,>=3.19.6, but you have protobuf 5.29.6 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have p

In [8]:
!pip install -q diffusers accelerate scipy

In [9]:
# ================================================================
# IMPORTS & SETUP
# ================================================================
import json
import re
import os
import time
import asyncio
import warnings
import numpy as np
import torch
import soundfile as sf
import nest_asyncio
import edge_tts
from pydub import AudioSegment
from IPython.display import Audio, display, HTML
import google.generativeai as genai

nest_asyncio.apply()
warnings.filterwarnings("ignore", category=FutureWarning)

os.makedirs("audio_segments", exist_ok=True)
os.makedirs("audio_segments_parler", exist_ok=True)
os.makedirs("sfx", exist_ok=True)
os.makedirs("final_output", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("All imports successful.")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
All imports successful.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# ================================================================
# GEMINI API KEY
# ================================================================
# OPTION 1: Colab Secrets (recommended)
#   Sidebar -> Key icon -> Add 'GEMINI_API_KEY'
# OPTION 2: Paste below (remove before sharing)
# ================================================================
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("API key loaded from Colab Secrets.")
except:
    GEMINI_API_KEY = ""  # <-- Replace this
    print("Using hardcoded key. Remove before sharing!")

genai.configure(api_key=GEMINI_API_KEY)
llm_model = genai.GenerativeModel('gemini-2.5-flash')

# Quick test
resp = llm_model.generate_content("Respond with only: API working")
print(f"Gemini: {resp.text.strip()}")

Using hardcoded key. Remove before sharing!
Gemini: API working


---
## Sample Input

In [11]:
sample_text = """
The old library was silent except for the ticking of the grandfather clock.

"I trusted you," Maria said softly, her eyes glistening. "I trusted you with everything."

The door creaked open slowly. Thomas stepped inside, his coat dripping with rain.

"How could you do this to us?" he shouted, slamming his fist on the table.

A cold wind howled through the broken window.

"Maybe it's not your fault," she whispered, turning away. "Maybe it was never anyone's fault."

Thomas sighed heavily. "I'm sorry, Maria. I truly am."

"Sorry doesn't fix what's broken," she replied, her voice barely audible.

The clock struck midnight. Somewhere in the distance, a dog barked.

"Then let me try," Thomas said gently, reaching for her hand. "Let me make it right."

Maria looked at him for a long moment. "I want to believe you," she said. "I really do."
"""

print(f"Sample text loaded: {len(sample_text)} characters")

Sample text loaded: 847 characters


---
# STEP 1 — LLM-Based Text Analysis

Gemini analyzes the full text in a single call and returns structured JSON with:
- Character detection and tracking
- Dialogue / narration / SFX segmentation
- Emotion, speaking style, gender, age for each segment

This replaces all the regex functions from the original notebook.

In [12]:
ANALYSIS_PROMPT = """
You are a text analysis engine for an audiobook generation system.

Analyze the following text and break it into an ordered list of segments.
Each segment is one of: "dialogue", "narration", or "sfx" (sound effect).

For EACH segment, provide:

If type is "dialogue":
- speaker: The character's name (e.g., "Maria", "Thomas"). Track speakers consistently.
- text: The exact dialogue text (without surrounding quotes).
- emotion: One of [neutral, happy, sad, angry, fearful, surprised, disgusted, tender, hopeful, resigned]
- gender: "male" or "female"
- age_group: "child", "young_adult", "adult", "elderly"
- accent: accent if explicitly mentioned, else null
- speaking_style: One of [whisper, soft, normal, loud, shout]

If type is "narration":
- text: The narration text.
- mood: The overall mood (e.g., "tense", "calm", "melancholic")

If type is "sfx":
- effect: Short label (e.g., "door_creak", "wind_howl", "clock_tick")
- description: Brief description of the sound

IMPORTANT RULES:
1. Maintain speaker identity across the entire text.
2. A single sentence can produce MULTIPLE segments.
   '"I trusted you," Maria said softly.' produces:
   - dialogue segment with text "I trusted you,"
   - narration segment with text "Maria said softly."
3. Extract SFX from narration when environmental sounds are described.
   Keep the narration text as well.
4. Emotions should reflect CONTENT and CONTEXT, not just punctuation.
5. Keep segments in exact reading order.

Respond ONLY with valid JSON. No markdown, no backticks.
{
  "characters": [
    {"name": "...", "gender": "...", "age_group": "...", "description": "..."}
  ],
  "segments": [ ... ]
}

TEXT TO ANALYZE:
---
"""


def analyze_text_with_llm(text, model):
    """Send text to Gemini and get structured analysis back."""
    prompt = ANALYSIS_PROMPT + text + "\n---"
    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=0.1,
            response_mime_type="application/json",
        )
    )
    return json.loads(response.text)


def analyze_long_text(text, model, max_chars=3000):
    """Chunk long texts and analyze each chunk, maintaining character consistency."""
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) + 2 > max_chars and current:
            chunks.append(current.strip())
            current = para
        else:
            current += "\n\n" + para
    if current.strip():
        chunks.append(current.strip())

    all_segments, all_characters = [], {}
    for i, chunk in enumerate(chunks):
        print(f"  Analyzing chunk {i+1}/{len(chunks)}...")
        if all_characters:
            chunk = f"Known characters: {json.dumps(list(all_characters.values()))}\n\n{chunk}"
        result = analyze_text_with_llm(chunk, model)
        for char in result.get('characters', []):
            all_characters[char['name']] = char
        all_segments.extend(result.get('segments', []))

    return {'characters': list(all_characters.values()), 'segments': all_segments}


print("Step 1 functions defined.")

Step 1 functions defined.


In [13]:
# Run analysis
print("Analyzing text with Gemini...")
analysis = analyze_long_text(sample_text, llm_model)

print(f"\nDetected {len(analysis['characters'])} characters:")
for c in analysis['characters']:
    print(f"  {c['name']} — {c.get('gender','?')}, {c.get('age_group','?')}")

print(f"\nGenerated {len(analysis['segments'])} segments:")
for i, seg in enumerate(analysis['segments']):
    t = seg['type']
    if t == 'dialogue':
        print(f"  [{i:2d}] DIALOGUE  {seg.get('speaker','?'):10s} | {seg.get('emotion','?'):10s} | {seg.get('speaking_style','?'):8s} | \"{seg['text'][:50]}\"")
    elif t == 'narration':
        print(f"  [{i:2d}] NARRATION  {'':10s} | {seg.get('mood','?'):10s} | {'':8s} | \"{seg['text'][:50]}\"")
    elif t == 'sfx':
        print(f"  [{i:2d}] SFX        {'':10s} | {seg.get('effect','?')}")

Analyzing text with Gemini...
  Analyzing chunk 1/1...

Detected 2 characters:
  Maria — female, adult
  Thomas — male, adult

Generated 32 segments:
  [ 0] NARRATION             | calm       |          | "The old library was silent except for the ticking "
  [ 1] SFX                   | clock_tick
  [ 2] DIALOGUE  Maria      | sad        | soft     | "I trusted you,"
  [ 3] NARRATION             | melancholic |          | "Maria said softly, her eyes glistening."
  [ 4] DIALOGUE  Maria      | sad        | soft     | "I trusted you with everything."
  [ 5] SFX                   | door_creak
  [ 6] NARRATION             | tense      |          | "The door creaked open slowly."
  [ 7] NARRATION             | tense      |          | "Thomas stepped inside, his coat dripping with rain"
  [ 8] DIALOGUE  Thomas     | angry      | shout    | "How could you do this to us?"
  [ 9] NARRATION             | angry      |          | "he shouted, slamming his fist on the table."
  [10] SFX           

---
# STEP 2 — Voice Mapping & Profile Database

Maps each character to a distinct voice and maps emotions to prosody parameters.

Two voice backends:
- **edge-tts voices** — for fast synthesis (emotion via rate/pitch/volume)
- **Parler-TTS Expresso** — for natural emotional speech (emotion via text description prompts: "Jerry speaks with a sad tone")

In [ ]:

# VOICE PROFILES — edge-tts voices by gender/age

EDGE_VOICE_PROFILES = {
    ("female", "adult"):       ["en-US-JennyNeural", "en-US-AriaNeural", "en-GB-SoniaNeural"],
    ("female", "young_adult"): ["en-US-AriaNeural", "en-US-JennyNeural"],
    ("female", "elderly"):     ["en-US-JennyNeural"],
    ("female", "child"):       ["en-US-AnaNeural"],
    ("male", "adult"):         ["en-US-GuyNeural", "en-US-DavisNeural", "en-GB-RyanNeural"],
    ("male", "young_adult"):   ["en-US-DavisNeural", "en-US-GuyNeural"],
    ("male", "elderly"):       ["en-US-GuyNeural"],
    ("male", "child"):         ["en-US-AnaNeural"],
}
NARRATOR_VOICE = "en-US-DavisNeural"

# Parler-TTS Expresso speaker names (from the model card)
# Jerry (male), Thomas (male), Elisabeth (female), Talia (female)
PARLER_SPEAKERS = {
    "female": ["Elisabeth", "Talia"],
    "male": ["Jerry", "Thomas"],
}

# Emotion -> prosody for edge-tts
EMOTION_PROSODY = {
    "neutral":   {"rate": "+0%",   "pitch": "+0Hz",  "volume": "+0%"},
    "happy":     {"rate": "+10%",  "pitch": "+20Hz", "volume": "+5%"},
    "sad":       {"rate": "-15%",  "pitch": "-15Hz", "volume": "-15%"},
    "angry":     {"rate": "+10%",  "pitch": "+10Hz", "volume": "+25%"},
    "fearful":   {"rate": "+20%",  "pitch": "+15Hz", "volume": "-10%"},
    "surprised": {"rate": "+15%",  "pitch": "+30Hz", "volume": "+10%"},
    "disgusted": {"rate": "-5%",   "pitch": "-10Hz", "volume": "+5%"},
    "tender":    {"rate": "-10%",  "pitch": "-5Hz",  "volume": "-15%"},
    "hopeful":   {"rate": "+5%",   "pitch": "+10Hz", "volume": "+0%"},
    "resigned":  {"rate": "-10%",  "pitch": "-10Hz", "volume": "-10%"},
}

STYLE_PROSODY = {
    "whisper":  {"rate": "-20%", "pitch": "-5Hz",  "volume": "-35%"},
    "soft":     {"rate": "-8%",  "pitch": "-3Hz",  "volume": "-20%"},
    "normal":   {"rate": "+0%",  "pitch": "+0Hz",  "volume": "+0%"},
    "loud":     {"rate": "+5%",  "pitch": "+5Hz",  "volume": "+15%"},
    "shout":    {"rate": "+10%", "pitch": "+15Hz", "volume": "+30%"},
}

# Emotion -> Parler-TTS description keyword
# Parler Expresso supports: happy, confused, sad, laughing, whisper, emphasis, default
EMOTION_TO_PARLER = {
    "neutral":   "default",
    "happy":     "happy",
    "sad":       "sad",
    "angry":     "emphasis",      # no angry, use emphasis for intensity
    "fearful":   "confused",
    "surprised": "confused",
    "disgusted": "emphasis",
    "tender":    "sad",           # tender maps closest to soft/sad
    "hopeful":   "happy",
    "resigned":  "sad",
}

print("Voice profiles loaded.")

Voice profiles loaded.


In [ ]:

# VOICE MAPPER


class VoiceMapper:
    def __init__(self):
        self.edge_voices = {}     # character -> edge-tts voice name
        self.parler_voices = {}   # character -> Parler speaker name
        self.used_edge = set()
        self.used_parler = {"female": 0, "male": 0}

    def assign(self, name, gender="female", age_group="adult", accent=None):
        """Assign both edge-tts and Parler voices to a character."""
        if name in self.edge_voices:
            return

        # edge-tts assignment
        if accent and "british" in accent.lower():
            ev = "en-GB-SoniaNeural" if gender == "female" else "en-GB-RyanNeural"
        elif accent and "indian" in accent.lower():
            ev = "en-IN-NeerjaNeural" if gender == "female" else "en-IN-PrabhatNeural"
        else:
            key = (gender or "female", age_group or "adult")
            avail = EDGE_VOICE_PROFILES.get(key, EDGE_VOICE_PROFILES[("female", "adult")])
            ev = next((v for v in avail if v not in self.used_edge), avail[0])
        self.edge_voices[name] = ev
        self.used_edge.add(ev)

        # Parler-TTS assignment
        g = gender or "female"
        speakers = PARLER_SPEAKERS.get(g, PARLER_SPEAKERS["female"])
        idx = self.used_parler[g] % len(speakers)
        self.parler_voices[name] = speakers[idx]
        self.used_parler[g] += 1

    def get_edge_config(self, segment):
        """Get edge-tts voice + prosody for a segment."""
        if segment['type'] == 'sfx':
            return None

        if segment['type'] == 'dialogue':
            voice = self.edge_voices.get(segment.get('speaker'), NARRATOR_VOICE)
            emo = segment.get('emotion', 'neutral')
            sty = segment.get('speaking_style', 'normal')
        else:
            voice = NARRATOR_VOICE
            emo, sty = 'neutral', 'normal'

        ep = EMOTION_PROSODY.get(emo, EMOTION_PROSODY['neutral'])
        sp = STYLE_PROSODY.get(sty, STYLE_PROSODY['normal'])

        def combine(v1, v2):
            n1 = int(re.search(r'[+-]?\d+', v1).group())
            n2 = int(re.search(r'[+-]?\d+', v2).group())
            unit = 'Hz' if 'Hz' in v1 else '%'
            val = max(-50, min(50, n1 + n2))
            return f"{'+' if val >= 0 else ''}{val}{unit}"

        prosody = {k: combine(ep[k], sp[k]) for k in ['rate', 'pitch', 'volume']}
        return {"voice": voice, "prosody": prosody}

    def get_parler_description(self, segment):
        """Build a Parler-TTS natural language description for a segment."""
        if segment['type'] == 'sfx':
            return None, None

        if segment['type'] == 'dialogue':
            speaker_name = self.parler_voices.get(segment.get('speaker'), 'Jerry')
            emo = segment.get('emotion', 'neutral')
            style = segment.get('speaking_style', 'normal')
        else:
            speaker_name = 'Jerry'  # narrator
            emo, style = 'neutral', 'normal'

        parler_emo = EMOTION_TO_PARLER.get(emo, 'default')

        # Build natural language description
        speed_map = {'whisper': 'slowly', 'soft': 'at a slow pace', 'normal': 'at a moderate speed',
                     'loud': 'at a slightly fast pace', 'shout': 'quickly'}
        speed = speed_map.get(style, 'at a moderate speed')

        if parler_emo == 'default':
            desc = f"{speaker_name} speaks {speed} with high quality audio."
        elif parler_emo == 'whisper':
            desc = f"{speaker_name} whispers {speed} with high quality audio."
        elif parler_emo == 'emphasis':
            desc = f"{speaker_name} speaks with *emphasis* {speed} with high quality audio."
        else:
            desc = f"{speaker_name} speaks with a {parler_emo} tone {speed} with high quality audio."

        return speaker_name, desc


# Initialize and assign voices
mapper = VoiceMapper()
for char in analysis['characters']:
    mapper.assign(char['name'], char.get('gender'), char.get('age_group'))

print("Voice assignments:")
for name in mapper.edge_voices:
    print(f"  {name:12s} | edge-tts: {mapper.edge_voices[name]:25s} | Parler: {mapper.parler_voices[name]}")
print(f"  {'Narrator':12s} | edge-tts: {NARRATOR_VOICE:25s} | Parler: Jerry")

Voice assignments:
  Maria        | edge-tts: en-US-JennyNeural         | Parler: Elisabeth
  Thomas       | edge-tts: en-US-GuyNeural           | Parler: Jerry
  Narrator     | edge-tts: en-US-DavisNeural         | Parler: Jerry


---
# STEP 3 — TTS Synthesis

Two backends. Run **both** and compare:

**3A. edge-tts** — Fast, cloud-based, good baseline. Emotion expressed through prosody.

**3B. Parler-TTS Expresso** — Slower but more natural sounding. Emotion controlled through natural language prompts like *"Elisabeth speaks with a sad tone slowly with high quality audio."* This is the key upgrade for naturalness.

In [ ]:


# 3A. edge-tts SYNTHESIS (with error handling fixes)


async def synthesize_edge(segments, mapper):
    """Synthesize all segments with edge-tts. Handles short text and prosody errors."""
    audio_files = []

    for i, seg in enumerate(segments):
        config = mapper.get_edge_config(seg)
        text = seg.get('text', '').strip()

        if config is None or not text:
            audio_files.append(None)
            continue

        # FIX: Skip segments that are too short for edge-tts
        # edge-tts needs at least a few words to synthesize properly
        if len(text.split()) < 2:
            # Pad very short text with a natural pause word
            text = text + "..."

        # FIX: Remove problematic characters that cause edge-tts to fail
        text = text.replace('"', '').replace('"', '').replace('"', '')
        text = text.replace('\u2014', '-').replace('\u2013', '-')

        # FIX: Clamp prosody values to safe ranges for edge-tts
        prosody = config['prosody'].copy()

        # edge-tts rate must be like "+20%" or "-10%" — clamp to ±40%
        for key in ['rate', 'volume']:
            val = int(re.search(r'[+-]?\d+', prosody[key]).group())
            val = max(-40, min(40, val))
            prosody[key] = f"{'+' if val >= 0 else ''}{val}%"

        # edge-tts pitch: use Hz but keep in safe range ±30Hz
        pitch_val = int(re.search(r'[+-]?\d+', prosody['pitch']).group())
        pitch_val = max(-30, min(30, pitch_val))
        prosody['pitch'] = f"{'+' if pitch_val >= 0 else ''}{pitch_val}Hz"

        path = f"audio_segments/edge_{i:03d}.mp3"

        # Try synthesis with retry on failure
        success = False
        for attempt in range(3):
            try:
                comm = edge_tts.Communicate(
                    text=text,
                    voice=config['voice'],
                    rate=prosody['rate'],
                    volume=prosody['volume'],
                    pitch=prosody['pitch'],
                )
                await comm.save(path)

                # Verify the file was actually created and has content
                if os.path.exists(path) and os.path.getsize(path) > 100:
                    success = True
                    break
                else:
                    os.remove(path) if os.path.exists(path) else None
            except Exception as e:
                if attempt < 2:
                    # Retry with neutral prosody
                    prosody = {"rate": "+0%", "pitch": "+0Hz", "volume": "+0%"}
                    continue
                break

        if success:
            audio_files.append(path)
            label = seg.get('speaker', 'Narrator') if seg['type'] == 'dialogue' else 'Narrator'
            emo = seg.get('emotion', seg.get('mood', '-'))
            print(f"  [{i:2d}] {seg['type']:10s} {label:10s} | {emo:10s} -> {path}")
        else:
            audio_files.append(None)
            print(f"  [{i:2d}] SKIPPED    (text too short or edge-tts error for: \"{text[:40]}\")")

    return audio_files


print("Synthesizing with edge-tts (with error handling)...")
edge_files = asyncio.run(synthesize_edge(analysis['segments'], mapper))
valid = sum(1 for f in edge_files if f)
print(f"\nGenerated {valid}/{len(edge_files)} audio files successfully.")
if valid < len(edge_files):
    print(f"  ({len(edge_files) - valid} segments skipped — these are usually very short narration tags)")


Synthesizing with edge-tts (with error handling)...
  [ 0] SKIPPED    (text too short or edge-tts error for: "The old library was silent except for th")
  [ 2] dialogue   Maria      | sad        -> audio_segments/edge_002.mp3
  [ 3] SKIPPED    (text too short or edge-tts error for: "Maria said softly, her eyes glistening.")
  [ 4] dialogue   Maria      | sad        -> audio_segments/edge_004.mp3
  [ 6] SKIPPED    (text too short or edge-tts error for: "The door creaked open slowly.")
  [ 7] SKIPPED    (text too short or edge-tts error for: "Thomas stepped inside, his coat dripping")
  [ 8] dialogue   Thomas     | angry      -> audio_segments/edge_008.mp3
  [ 9] SKIPPED    (text too short or edge-tts error for: "he shouted, slamming his fist on the tab")
  [12] SKIPPED    (text too short or edge-tts error for: "A cold wind howled through the broken wi")
  [13] dialogue   Maria      | resigned   -> audio_segments/edge_013.mp3
  [14] SKIPPED    (text too short or edge-tts error for: "she 

In [ ]:

# 3B. PARLER-TTS EXPRESSO SYNTHESIS (more natural, emotional)

# This model takes ~1-2 min to load. Generation is slower but
# significantly more natural than edge-tts.


from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

print("Loading Parler-TTS Expresso model (this takes 1-2 minutes)...")
parler_model = ParlerTTSForConditionalGeneration.from_pretrained(
    "parler-tts/parler-tts-mini-expresso"
).to(DEVICE)
parler_tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler-tts-mini-expresso")
print("Parler-TTS loaded.")

Loading Parler-TTS Expresso model (this takes 1-2 minutes)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
  "_name_or_path": "google/flan-t5-base",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "n

Parler-TTS loaded.


In [18]:
def synthesize_parler_segment(text, description, model, tokenizer, device, output_path):
    """Synthesize a single segment with Parler-TTS."""
    input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
    prompt_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

    with torch.no_grad():
        generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_ids)

    audio_arr = generation.cpu().numpy().squeeze()
    sf.write(output_path, audio_arr, model.config.sampling_rate)
    return output_path


print("Synthesizing with Parler-TTS Expresso...")
parler_files = []

for i, seg in enumerate(analysis['segments']):
    speaker_name, desc = mapper.get_parler_description(seg)
    text = seg.get('text', '').strip()

    if desc is None or not text:
        parler_files.append(None)
        continue

    path = f"audio_segments_parler/parler_{i:03d}.wav"
    try:
        synthesize_parler_segment(text, desc, parler_model, parler_tokenizer, DEVICE, path)
        parler_files.append(path)
        label = seg.get('speaker', 'Narrator') if seg['type'] == 'dialogue' else 'Narrator'
        emo = seg.get('emotion', seg.get('mood', '-'))
        print(f"  [{i:2d}] {label:10s} | {emo:10s} | {desc[:60]}")
    except Exception as e:
        print(f"  [{i:2d}] ERROR: {e}")
        parler_files.append(None)

print(f"\nGenerated {sum(1 for f in parler_files if f)} Parler segments.")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Synthesizing with Parler-TTS Expresso...
  [ 0] Narrator   | calm       | Jerry speaks at a moderate speed with high quality audio.
  [ 2] Maria      | sad        | Elisabeth speaks with a sad tone at a slow pace with high qu
  [ 3] Narrator   | melancholic | Jerry speaks at a moderate speed with high quality audio.
  [ 4] Maria      | sad        | Elisabeth speaks with a sad tone at a slow pace with high qu
  [ 6] Narrator   | tense      | Jerry speaks at a moderate speed with high quality audio.
  [ 7] Narrator   | tense      | Jerry speaks at a moderate speed with high quality audio.
  [ 8] Thomas     | angry      | Jerry speaks with *emphasis* quickly with high quality audio
  [ 9] Narrator   | angry      | Jerry speaks at a moderate speed with high quality audio.
  [12] Narrator   | bleak      | Jerry speaks at a moderate speed with high quality audio.
  [13] Maria      | resigned   | Elisabeth speaks with a sad tone slowly with high quality au
  [14] Narrator   | melancholic | Je

---
# STEP 4 — SFX / Background Ambience

Generates simple ambient sound effects using synthesized tones and noise.
For a production system you would pull from a library like freesound.org,
but for the course project this demonstrates the concept.

In [ ]:

# STEP 4 — SFX with AudioLDM (AI-generated realistic sound effects)
# Uses AudioLDM (diffusion model) to generate sound effects from
# text descriptions like "a wooden door creaking open slowly".
# Falls back to a simpler method if AudioLDM fails to load.


import scipy
from diffusers import AudioLDMPipeline

print("Loading AudioLDM model for SFX generation (takes ~1 min)...")
try:
    sfx_pipe = AudioLDMPipeline.from_pretrained(
        "cvssp/audioldm-s-full-v2", torch_dtype=torch.float16
    )
    sfx_pipe = sfx_pipe.to(DEVICE)
    USE_AUDIOLDM = True
    print("AudioLDM loaded successfully.")
except Exception as e:
    print(f"AudioLDM failed to load: {e}")
    print("Will fall back to procedural SFX.")
    USE_AUDIOLDM = False


# SFX effect label -> descriptive prompt for AudioLDM
SFX_PROMPTS = {
    "door_creak":     "a wooden door slowly creaking open in a quiet room",
    "wind_howl":      "cold wind howling through a broken window at night",
    "clock_tick":     "a grandfather clock ticking steadily in a silent room",
    "clock_strike":   "a large clock striking midnight with deep resonant chimes",
    "thunder":        "distant thunder rumbling across the sky",
    "footsteps":      "slow footsteps on a hardwood floor",
    "rain":           "heavy rain falling on a rooftop",
    "dog_bark":       "a dog barking in the distance at night",
    "table_slam":     "a fist slamming hard on a wooden table",
    "glass_break":    "glass shattering on a hard floor",
    "fire_crackle":   "a warm fire crackling in a fireplace",
    "sigh":           "a person sighing heavily",
}


def generate_sfx_audioldm(effect_name, duration_s=3, output_dir="sfx"):
    """Generate realistic SFX using AudioLDM diffusion model."""
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f"{effect_name}.wav")

    # Get descriptive prompt, or build one from the effect name
    prompt = SFX_PROMPTS.get(effect_name)
    if prompt is None:
        # Convert snake_case to natural language
        prompt = effect_name.replace("_", " ") + ", high quality sound effect"

    if USE_AUDIOLDM:
        try:
            audio = sfx_pipe(
                prompt,
                num_inference_steps=15,       # fewer steps = faster, still decent
                audio_length_in_s=duration_s,
                negative_prompt="low quality, distorted, music, speech",
            ).audios[0]

            scipy.io.wavfile.write(path, rate=16000, data=audio)
            print(f"  [AudioLDM] {effect_name}: \"{prompt}\" -> {path}")
            return path
        except Exception as e:
            print(f"  [AudioLDM] Error for {effect_name}: {e}, falling back to procedural")

    # Fallback: improved procedural SFX (better than pure sine waves)
    return generate_sfx_procedural(effect_name, duration_s, output_dir)


def generate_sfx_procedural(effect_name, duration_s=3, output_dir="sfx"):
    """Improved procedural fallback SFX."""
    os.makedirs(output_dir, exist_ok=True)
    sr = 16000
    samples = int(sr * duration_s)
    t = np.linspace(0, duration_s, samples)
    path = os.path.join(output_dir, f"{effect_name}.wav")

    if "creak" in effect_name:
        freq = np.linspace(60, 300, samples) + 50 * np.sin(2 * np.pi * 3 * t)
        audio = 0.4 * np.sin(2 * np.pi * freq * t)
        audio *= np.random.uniform(0.5, 1.0, samples)
        env = np.concatenate([np.linspace(0, 1, samples//4),
                              np.ones(samples//2),
                              np.linspace(1, 0, samples - samples//4 - samples//2)])
        audio *= env

    elif "wind" in effect_name:
        noise = np.random.randn(samples)
        kernel_size = 800
        kernel = np.hanning(kernel_size)
        kernel /= kernel.sum()
        audio = np.convolve(noise, kernel, mode='same')
        mod = 0.5 + 0.5 * np.sin(2 * np.pi * 0.15 * t)
        audio *= mod * 0.6

    elif "clock" in effect_name and "strike" not in effect_name:
        audio = np.zeros(samples)
        interval = int(sr * 0.6)
        for start in range(0, samples, interval):
            tick_len = min(int(sr * 0.03), samples - start)
            tick_t = np.linspace(0, 0.03, tick_len)
            tick = 0.7 * np.sin(2 * np.pi * 1200 * tick_t) * np.exp(-tick_t * 100)
            audio[start:start + tick_len] += tick

    elif "strike" in effect_name or "chime" in effect_name:
        audio = np.zeros(samples)
        for n in range(3):
            start = int(sr * n * 0.8)
            chime_len = min(int(sr * 1.5), samples - start)
            chime_t = np.linspace(0, 1.5, chime_len)
            chime = (0.5 * np.sin(2 * np.pi * 220 * chime_t) +
                     0.3 * np.sin(2 * np.pi * 440 * chime_t)) * np.exp(-chime_t * 2)
            audio[start:start + chime_len] += chime

    elif "bark" in effect_name or "dog" in effect_name:
        audio = np.zeros(samples)
        bark_positions = [int(sr * 0.2), int(sr * 0.7), int(sr * 1.3), int(sr * 1.8)]
        for pos in bark_positions:
            if pos >= samples:
                break
            bark_len = min(int(sr * 0.12), samples - pos)
            bark_t = np.linspace(0, 0.12, bark_len)
            bark = (0.6 * np.sin(2 * np.pi * 350 * bark_t) +
                    0.3 * np.sin(2 * np.pi * 700 * bark_t)) * np.exp(-bark_t * 25)
            audio[pos:pos + bark_len] += bark

    elif "slam" in effect_name or "fist" in effect_name or "table" in effect_name:
        audio = np.zeros(samples)
        impact_len = min(int(sr * 0.15), samples)
        impact_t = np.linspace(0, 0.15, impact_len)
        impact = (np.random.randn(impact_len) * 0.5 +
                  0.8 * np.sin(2 * np.pi * 80 * impact_t)) * np.exp(-impact_t * 30)
        audio[:impact_len] = impact

    elif "thunder" in effect_name:
        noise = np.random.randn(samples)
        kernel = np.hanning(3000)
        kernel /= kernel.sum()
        audio = np.convolve(noise, kernel, mode='same')
        audio *= np.exp(-t * 1.5) * 2.0

    else:
        audio = 0.02 * np.random.randn(samples)

    audio = audio / (np.max(np.abs(audio)) + 1e-8) * 0.7
    audio_int16 = (audio * 32767).astype(np.int16)
    scipy.io.wavfile.write(path, sr, audio_int16)
    print(f"  [Procedural] {effect_name} -> {path}")
    return path


# Generate all SFX from the analysis
sfx_files = {}
print("\nGenerating SFX:")
for seg in analysis['segments']:
    if seg['type'] == 'sfx':
        effect = seg.get('effect', 'ambient')
        if effect not in sfx_files:
            sfx_files[effect] = generate_sfx_audioldm(effect)

print(f"Generated {len(sfx_files)} SFX files.")

# Preview SFX
for name, path in sfx_files.items():
    print(f"\n  {name}:")
    display(Audio(path))

Loading AudioLDM model for SFX generation (takes ~1 min)...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

The AudioLDMPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


AudioLDM loaded successfully.

Generating SFX:


  0%|          | 0/15 [00:00<?, ?it/s]

  [AudioLDM] clock_tick: "a grandfather clock ticking steadily in a silent room" -> sfx/clock_tick.wav


... (14 outputs left out)


  wind_howl:



  sigh_heavy:



  clock_chime:



  dog_bark:


---
# STEP 5 — Audio Stitching & Post-processing

Combines all segments (dialogue, narration, SFX) into a final audiobook with proper pacing.

In [ ]:

# AUDIO STITCHER


def stitch_audiobook(segments, audio_files, sfx_files_dict, output_path,
                     dialogue_pause=400, narration_pause=600, sfx_volume_db=-10):
    """
    Stitch all segments into a final audiobook.
    SFX are overlaid at reduced volume behind the next segment.
    """
    combined = AudioSegment.silent(duration=300)  # opening silence

    for i, (seg, filepath) in enumerate(zip(segments, audio_files)):
        if seg['type'] == 'sfx':
            effect = seg.get('effect', 'ambient')
            sfx_path = sfx_files_dict.get(effect)
            if sfx_path:
                try:
                    sfx_audio = AudioSegment.from_wav(sfx_path) + sfx_volume_db
                    combined += sfx_audio
                except:
                    combined += AudioSegment.silent(duration=200)
            continue

        if filepath is None:
            continue

        try:
            if filepath.endswith('.mp3'):
                seg_audio = AudioSegment.from_mp3(filepath)
            else:
                seg_audio = AudioSegment.from_wav(filepath)

            combined += seg_audio

            # Add pause
            pause = dialogue_pause if seg['type'] == 'dialogue' else narration_pause
            combined += AudioSegment.silent(duration=pause)
        except Exception as e:
            print(f"  Warning: skipped segment {i}: {e}")

    combined += AudioSegment.silent(duration=500)  # closing silence
    combined.export(output_path, format="mp3", bitrate="192k")

    dur = len(combined) / 1000
    print(f"  Saved: {output_path} ({dur:.1f}s / {dur/60:.1f}min)")
    return output_path


# Stitch edge-tts version
print("Stitching edge-tts audiobook...")
edge_output = stitch_audiobook(
    analysis['segments'], edge_files, sfx_files,
    "final_output/audiobook_edge.mp3"
)

# Stitch Parler-TTS version
print("Stitching Parler-TTS audiobook...")
parler_output = stitch_audiobook(
    analysis['segments'], parler_files, sfx_files,
    "final_output/audiobook_parler.mp3"
)

print("\n--- edge-tts version ---")
display(Audio(edge_output))
print("\n--- Parler-TTS Expresso version (more natural) ---")
display(Audio(parler_output))

Stitching edge-tts audiobook...
  Saved: final_output/audiobook_edge.mp3 (57.5s / 1.0min)
Stitching Parler-TTS audiobook...
  Saved: final_output/audiobook_parler.mp3 (97.6s / 1.6min)

--- edge-tts version ---



--- Parler-TTS Expresso version (more natural) ---


---
# STEP 6 — Human-in-the-Loop Validation (Gradio)

An interactive interface where you can:
- See the LLM's analysis for each segment
- Change the speaker, emotion, or style
- Re-synthesize individual segments
- Preview and compare audio

In [21]:
pip install uvicorn==0.23.2

  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.42.0
    Uninstalling uvicorn-0.42.0:
      Successfully uninstalled uvicorn-0.42.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.27.1 requires uvicorn<1.0.0,>=0.34.0, but you have uvicorn 0.23.2 which is incompatible.
python-fasthtml 0.12.50 requires uvicorn[standard]>=0.30, but you have uvicorn 0.23.2 which is incompatible.
mcp 1.26.0 requires uvicorn>=0.31.1; sys_platform != "emscripten", but you have uvicorn 0.23.2 which is incompatible.


In [ ]:

# FIX: Remove uvloop completely — it breaks Gradio in Colab
# Run this cell ONCE, then run the Step 6 cell below

import subprocess
subprocess.run(["pip", "uninstall", "uvloop", "-y"], capture_output=True)

# Force Python to forget uvloop was ever imported
import sys
for mod_name in list(sys.modules.keys()):
    if "uvloop" in mod_name:
        del sys.modules[mod_name]

# Reset event loop policy to default
import asyncio
asyncio.set_event_loop_policy(None)

# Re-apply nest_asyncio on a clean default loop
import nest_asyncio
nest_asyncio.apply()

print("uvloop removed. Now run the Step 6 cell below.")

uvloop removed. Now run the Step 6 cell below.


In [ ]:

# STEP 6 — Human-in-the-Loop Validation (Gradio)

# FIX: Unset uvloop policy so Gradio's uvicorn can start properly


import asyncio
import gradio as gr
import pandas as pd

# THIS IS THE KEY FIX — uvloop breaks Gradio in Colab threads
asyncio.set_event_loop_policy(None)

# Build segment table
def build_segment_df(segments):
    rows = []
    for i, seg in enumerate(segments):
        if seg['type'] == 'dialogue':
            rows.append({"Idx": i, "Type": "dialogue", "Speaker": seg.get('speaker','?'),
                         "Emotion": seg.get('emotion','neutral'), "Style": seg.get('speaking_style','normal'),
                         "Text": seg['text'][:70]})
        elif seg['type'] == 'narration':
            rows.append({"Idx": i, "Type": "narration", "Speaker": "Narrator",
                         "Emotion": seg.get('mood','neutral'), "Style": "normal",
                         "Text": seg['text'][:70]})
        elif seg['type'] == 'sfx':
            rows.append({"Idx": i, "Type": "sfx", "Speaker": "-",
                         "Emotion": seg.get('effect','?'), "Style": "-",
                         "Text": seg.get('description', seg.get('effect',''))})
    return pd.DataFrame(rows)


def re_synthesize_segment(seg_idx, new_emotion, new_style):
    seg_idx = int(seg_idx)
    if seg_idx < 0 or seg_idx >= len(analysis['segments']):
        return None, f"Invalid index: {seg_idx}"

    seg = analysis['segments'][seg_idx]
    if seg['type'] == 'sfx':
        return None, "Cannot re-synthesize SFX."

    if seg['type'] == 'dialogue':
        seg['emotion'] = new_emotion
        seg['speaking_style'] = new_style
    else:
        seg['mood'] = new_emotion

    config = mapper.get_edge_config(seg)
    text = seg.get('text', '').strip()
    if not text:
        return None, "Empty text."
    if len(text.split()) < 2:
        text = text + "..."

    prosody = config['prosody'].copy()
    for key in ['rate', 'volume']:
        val = int(re.search(r'[+-]?\d+', prosody[key]).group())
        val = max(-40, min(40, val))
        prosody[key] = f"{'+' if val >= 0 else ''}{val}%"
    pitch_val = int(re.search(r'[+-]?\d+', prosody['pitch']).group())
    pitch_val = max(-30, min(30, pitch_val))
    prosody['pitch'] = f"{'+' if pitch_val >= 0 else ''}{pitch_val}Hz"

    path = f"audio_segments/resyn_{seg_idx:03d}.mp3"

    # Use a fresh event loop for edge-tts (avoids all async conflicts)
    async def _synth():
        comm = edge_tts.Communicate(
            text=text, voice=config['voice'],
            rate=prosody['rate'], volume=prosody['volume'], pitch=prosody['pitch'],
        )
        await comm.save(path)

    loop = asyncio.new_event_loop()
    loop.run_until_complete(_synth())
    loop.close()

    if seg_idx < len(edge_files):
        edge_files[seg_idx] = path

    label = seg.get('speaker', 'Narrator') if seg['type'] == 'dialogue' else 'Narrator'
    return path, f"Re-synthesized [{seg_idx}] {label}: emotion={new_emotion}, style={new_style}"


def restitch_audiobook():
    path = stitch_audiobook(
        analysis['segments'], edge_files, sfx_files,
        "final_output/audiobook_edited.mp3"
    )
    return path, "Full audiobook re-stitched with your edits."


segment_df = build_segment_df(analysis['segments'])
emotions = list(EMOTION_PROSODY.keys())
styles = list(STYLE_PROSODY.keys())

with gr.Blocks(title="Audiobook HITL Editor") as demo:
    gr.Markdown("# Audiobook Human-in-the-Loop Editor")
    gr.Markdown("Review the LLM analysis. Pick a segment, change emotion/style, re-synthesize.")

    gr.Dataframe(value=segment_df, label="Detected Segments", interactive=False)

    gr.Markdown("---")
    gr.Markdown("### Modify and Re-synthesize a Segment")
    with gr.Row():
        seg_input = gr.Number(label="Segment Index", value=0, precision=0,
                              minimum=0, maximum=len(analysis['segments'])-1)
        emo_input = gr.Dropdown(choices=emotions, label="New Emotion", value="neutral")
        style_input = gr.Dropdown(choices=styles, label="New Style", value="normal")

    resynth_btn = gr.Button("Re-synthesize This Segment", variant="primary")
    with gr.Row():
        audio_out = gr.Audio(label="Re-synthesized Audio", type="filepath")
        info_out = gr.Textbox(label="Status", interactive=False)

    resynth_btn.click(fn=re_synthesize_segment,
                      inputs=[seg_input, emo_input, style_input],
                      outputs=[audio_out, info_out])

    gr.Markdown("---")
    gr.Markdown("### Re-stitch Complete Audiobook")
    restitch_btn = gr.Button("Re-stitch Full Audiobook", variant="secondary")
    with gr.Row():
        full_audio_out = gr.Audio(label="Complete Edited Audiobook", type="filepath")
        full_info = gr.Textbox(label="Status", interactive=False)

    restitch_btn.click(fn=restitch_audiobook, outputs=[full_audio_out, full_info])

print("Launching Gradio interface...")
demo.launch(share=True, quiet=True)

Launching Gradio interface...
* Running on public URL: https://6ae6863e04ddb89878.gradio.live


---
# STEP 7 — Evaluation Metrics

| Metric | What it measures | Target |
|--------|-----------------|--------|
| **MOS (Mean Opinion Score)** | Audio naturalness (1-5 human rating) | > 3.5 |
| **WER (Word Error Rate)** | Speech clarity via Whisper transcription | < 15% |
| **Emotion F1** | Emotion classification accuracy | > 0.7 |
| **RTF (Real-Time Factor)** | Inference speed | < 1.0 |
| **Speaker Consistency** | Same character = same voice | Qualitative |

In [ ]:

# 7A. WER — Word Error Rate (using OpenAI Whisper)

# Transcribes each audio segment and compares to the original text.
# Lower WER = clearer speech.


import whisper
from jiwer import wer as compute_wer

print("Loading Whisper model for WER evaluation...")
whisper_model = whisper.load_model("base", device=DEVICE)
print("Whisper loaded.")


def evaluate_wer(segments, audio_files, whisper_model):
    """Compute WER for each dialogue/narration segment."""
    results = []
    total_ref_words = 0
    total_errors = 0

    for i, (seg, filepath) in enumerate(zip(segments, audio_files)):
        if filepath is None or seg['type'] == 'sfx':
            continue

        ref_text = seg['text'].strip()
        if not ref_text:
            continue

        try:
            # Transcribe
            result = whisper_model.transcribe(filepath, language="en")
            hyp_text = result['text'].strip()

            # Compute WER
            segment_wer = compute_wer(ref_text.lower(), hyp_text.lower())
            ref_words = len(ref_text.split())
            total_ref_words += ref_words
            total_errors += segment_wer * ref_words

            results.append({
                "idx": i,
                "reference": ref_text[:60],
                "hypothesis": hyp_text[:60],
                "wer": segment_wer
            })
        except Exception as e:
            print(f"  Segment {i} WER error: {e}")

    overall_wer = total_errors / total_ref_words if total_ref_words > 0 else 0
    return results, overall_wer


# Evaluate edge-tts
print("\nEvaluating WER for edge-tts output...")
edge_wer_results, edge_overall_wer = evaluate_wer(
    analysis['segments'], edge_files, whisper_model
)
print(f"  edge-tts overall WER: {edge_overall_wer:.2%}")

# Evaluate Parler-TTS
print("Evaluating WER for Parler-TTS output...")
parler_wer_results, parler_overall_wer = evaluate_wer(
    analysis['segments'], parler_files, whisper_model
)
print(f"  Parler-TTS overall WER: {parler_overall_wer:.2%}")

Loading Whisper model for WER evaluation...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 113MiB/s]


Whisper loaded.

Evaluating WER for edge-tts output...
  edge-tts overall WER: 11.11%
Evaluating WER for Parler-TTS output...
  Parler-TTS overall WER: 14.69%


In [27]:
# Show detailed WER results
print("\nDetailed WER — edge-tts:")
print(f"{'Idx':>4s} {'WER':>8s}  {'Reference':40s} {'Hypothesis':40s}")
print("-" * 96)
for r in edge_wer_results:
    flag = " !!" if r['wer'] > 0.3 else ""
    print(f"{r['idx']:4d} {r['wer']:7.1%}  {r['reference']:40s} {r['hypothesis']:40s}{flag}")

print(f"\nOverall edge-tts WER: {edge_overall_wer:.2%}")
print(f"Overall Parler-TTS WER: {parler_overall_wer:.2%}")


Detailed WER — edge-tts:
 Idx      WER  Reference                                Hypothesis                              
------------------------------------------------------------------------------------------------
   2   33.3%  I trusted you,                           I trusted you.                           !!
   4    0.0%  I trusted you with everything.           I trusted you with everything.          
   8    0.0%  How could you do this to us?             How could you do this to us?            
  13   20.0%  Maybe it's not your fault,               Maybe it's not your fault.              
  15    0.0%  Maybe it was never anyone's fault.       Maybe it was never anyone's fault.      
  18    0.0%  I'm sorry, Maria. I truly am.            I'm sorry, Maria. I truly am.           
  19   40.0%  Sorry doesn't fix what's broken,         Sorry, doesn't fix what's broken.        !!
  25   25.0%  Then let me try,                         Then let me try.                        
  27  

In [ ]:

# 7B. EMOTION F1 — Verify emotion classification accuracy

# Uses the j-hartmann emotion classifier to classify the *text*
# and compare against the LLM's emotion labels.


from transformers import pipeline
from sklearn.metrics import f1_score, classification_report

print("Loading emotion classifier...")
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=False,
    device=0 if DEVICE == "cuda" else -1
)

# Map between our emotion labels and the classifier's labels
# Classifier labels: anger, disgust, fear, joy, neutral, sadness, surprise
OUR_TO_CLASSIFIER = {
    "angry": "anger", "disgusted": "disgust", "fearful": "fear",
    "happy": "joy", "neutral": "neutral", "sad": "sadness",
    "surprised": "surprise", "tender": "sadness",
    "hopeful": "joy", "resigned": "sadness",
}

llm_labels = []
classifier_labels = []

for seg in analysis['segments']:
    if seg['type'] != 'dialogue':
        continue
    text = seg.get('text', '').strip()
    if not text:
        continue

    # LLM's label (mapped to classifier space)
    llm_emo = seg.get('emotion', 'neutral')
    llm_mapped = OUR_TO_CLASSIFIER.get(llm_emo, 'neutral')
    llm_labels.append(llm_mapped)

    # Classifier's prediction
    pred = emotion_classifier(text)[0]
    classifier_labels.append(pred['label'])

if llm_labels:
    all_labels = sorted(set(llm_labels + classifier_labels))
    f1 = f1_score(llm_labels, classifier_labels, average='weighted', labels=all_labels, zero_division=0)
    print(f"\nEmotion Classification Agreement (LLM vs Classifier):")
    print(f"  Weighted F1 Score: {f1:.3f}")
    print(f"\nDetailed Report:")
    print(classification_report(llm_labels, classifier_labels, labels=all_labels, zero_division=0))
else:
    print("No dialogue segments to evaluate.")

Loading emotion classifier...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Emotion Classification Agreement (LLM vs Classifier):
  Weighted F1 Score: 0.381

Detailed Report:
              precision    recall  f1-score   support

       anger       1.00      0.50      0.67         2
         joy       0.00      0.00      0.00         4
     neutral       0.00      0.00      0.00         0
     sadness       1.00      0.40      0.57         5
    surprise       0.00      0.00      0.00         0

    accuracy                           0.27        11
   macro avg       0.40      0.18      0.25        11
weighted avg       0.64      0.27      0.38        11



In [ ]:

# 7C. RTF — Real-Time Factor
# RTF = synthesis_time / audio_duration
# RTF < 1 means faster than real-time (good)


def measure_rtf_edge(text, voice="en-US-JennyNeural"):
    """Measure RTF for edge-tts."""
    path = "audio_segments/rtf_test_edge.mp3"

    async def _synth():
        comm = edge_tts.Communicate(text=text, voice=voice)
        await comm.save(path)

    start = time.time()
    asyncio.run(_synth())
    synth_time = time.time() - start

    audio = AudioSegment.from_mp3(path)
    audio_duration = len(audio) / 1000
    return synth_time / audio_duration if audio_duration > 0 else float('inf')


def measure_rtf_parler(text, description="Jerry speaks at a moderate speed with high quality audio."):
    """Measure RTF for Parler-TTS."""
    path = "audio_segments/rtf_test_parler.wav"

    start = time.time()
    synthesize_parler_segment(text, description, parler_model, parler_tokenizer, DEVICE, path)
    synth_time = time.time() - start

    audio = AudioSegment.from_wav(path)
    audio_duration = len(audio) / 1000
    return synth_time / audio_duration if audio_duration > 0 else float('inf')


test_text = "The quick brown fox jumped over the lazy dog and ran into the forest."

print("Measuring RTF...")
rtf_edge = measure_rtf_edge(test_text)
rtf_parler = measure_rtf_parler(test_text)

print(f"\n  edge-tts RTF:     {rtf_edge:.3f}  {'(real-time ok)' if rtf_edge < 1 else '(slower than real-time)'}")
print(f"  Parler-TTS RTF:   {rtf_parler:.3f}  {'(real-time ok)' if rtf_parler < 1 else '(slower than real-time)'}")

Measuring RTF...

  edge-tts RTF:     0.228  (real-time ok)
  Parler-TTS RTF:   1.995  (slower than real-time)


In [ ]:

# 7D. MOS — Mean Opinion Score (Automated Proxy)
# True MOS requires human listeners. Here we compute an automated
# proxy using signal-to-noise metrics and Whisper confidence.
# For your project report, also do a small human evaluation
# (5-10 listeners rating 1-5 on naturalness).


def automated_mos_proxy(audio_path, whisper_model):
    """Compute automated MOS proxy score (1-5 scale)."""
    try:
        if audio_path.endswith('.mp3'):
            audio = AudioSegment.from_mp3(audio_path)
        else:
            audio = AudioSegment.from_wav(audio_path)

        # Factor 1: Audio quality (dBFS - closer to 0 is better, very low is bad)
        dbfs = audio.dBFS
        quality_score = max(1, min(5, 5 + dbfs / 10))  # -50dB -> 0, -10dB -> 4

        # Factor 2: Whisper confidence (avg log probability)
        result = whisper_model.transcribe(audio_path, language="en")
        avg_logprob = np.mean([s.get('avg_logprob', -1) for s in result.get('segments', [{'avg_logprob': -1}])])
        confidence_score = max(1, min(5, 5 + avg_logprob * 3))

        # Factor 3: No-speech ratio (lower is better)
        no_speech = np.mean([s.get('no_speech_prob', 0.5) for s in result.get('segments', [{'no_speech_prob': 0.5}])])
        speech_score = max(1, min(5, 5 - no_speech * 5))

        # Weighted average
        mos = 0.3 * quality_score + 0.5 * confidence_score + 0.2 * speech_score
        return round(mos, 2), {
            "quality": round(quality_score, 2),
            "confidence": round(confidence_score, 2),
            "speech_clarity": round(speech_score, 2),
        }
    except Exception as e:
        return 0, {"error": str(e)}


print("Computing automated MOS proxy...")

# Sample a few segments for MOS
mos_edge_scores = []
mos_parler_scores = []

for i, seg in enumerate(analysis['segments']):
    if seg['type'] == 'sfx':
        continue
    if edge_files[i]:
        score, _ = automated_mos_proxy(edge_files[i], whisper_model)
        if score > 0:
            mos_edge_scores.append(score)
    if parler_files[i]:
        score, _ = automated_mos_proxy(parler_files[i], whisper_model)
        if score > 0:
            mos_parler_scores.append(score)

print(f"\n  edge-tts   avg MOS proxy: {np.mean(mos_edge_scores):.2f} / 5.00  (n={len(mos_edge_scores)})")
print(f"  Parler-TTS avg MOS proxy: {np.mean(mos_parler_scores):.2f} / 5.00  (n={len(mos_parler_scores)})")
print(f"\n  Note: For your project report, supplement this with a real human")
print(f"  MOS survey (5-10 listeners, rate each sample 1-5 on naturalness).")

Computing automated MOS proxy...

  edge-tts   avg MOS proxy: 3.72 / 5.00  (n=11)
  Parler-TTS avg MOS proxy: 3.42 / 5.00  (n=25)

  Note: For your project report, supplement this with a real human
  MOS survey (5-10 listeners, rate each sample 1-5 on naturalness).


In [ ]:

# FINAL EVALUATION SUMMARY


print("="*60)
print("       EVALUATION SUMMARY")
print("="*60)
print(f"")
print(f"  {'Metric':<30s} {'edge-tts':>12s} {'Parler-TTS':>12s}")
print(f"  {'-'*54}")
print(f"  {'WER (Word Error Rate)':<30s} {edge_overall_wer:>11.1%} {parler_overall_wer:>11.1%}")
print(f"  {'MOS Proxy (1-5)':<30s} {np.mean(mos_edge_scores):>12.2f} {np.mean(mos_parler_scores):>12.2f}")
print(f"  {'RTF (< 1 = real-time)':<30s} {rtf_edge:>12.3f} {rtf_parler:>12.3f}")
print(f"  {'Emotion F1 (LLM vs model)':<30s} {f1:>12.3f} {'(text-based)':>12s}")
print(f"  {'Characters detected':<30s} {len(analysis['characters']):>12d} {'-':>12s}")
print(f"  {'Segments generated':<30s} {len(analysis['segments']):>12d} {'-':>12s}")
print(f"")
print("="*60)

       EVALUATION SUMMARY

  Metric                             edge-tts   Parler-TTS
  ------------------------------------------------------
  WER (Word Error Rate)                11.1%       14.7%
  MOS Proxy (1-5)                        3.72         3.42
  RTF (< 1 = real-time)                 0.228        1.995
  Emotion F1 (LLM vs model)             0.381 (text-based)
  Characters detected                       2            -
  Segments generated                       32            -



---
## Export Everything

In [32]:
# Save analysis + metrics for the report
export = {
    "characters": analysis['characters'],
    "segments": analysis['segments'],
    "voice_assignments": {
        "edge_tts": mapper.edge_voices,
        "parler_tts": mapper.parler_voices,
        "narrator": NARRATOR_VOICE,
    },
    "evaluation": {
        "wer_edge": round(edge_overall_wer, 4),
        "wer_parler": round(parler_overall_wer, 4),
        "mos_edge": round(float(np.mean(mos_edge_scores)), 2),
        "mos_parler": round(float(np.mean(mos_parler_scores)), 2),
        "rtf_edge": round(rtf_edge, 3),
        "rtf_parler": round(rtf_parler, 3),
        "emotion_f1": round(f1, 3),
    }
}

with open("final_output/full_analysis.json", "w") as f:
    json.dump(export, f, indent=2)

print("Exported to final_output/full_analysis.json")
print("\nFiles in final_output/:")
for f in sorted(os.listdir("final_output")):
    size = os.path.getsize(f"final_output/{f}")
    print(f"  {f:40s} {size/1024:.0f} KB")

Exported to final_output/full_analysis.json

Files in final_output/:
  audiobook_edge.mp3                       1125 KB
  audiobook_parler.mp3                     2289 KB
  full_analysis.json                       6 KB


In [33]:
# Download files from Colab
from google.colab import files

print("Downloading outputs...")
files.download("final_output/audiobook_edge.mp3")
files.download("final_output/audiobook_parler.mp3")
files.download("final_output/full_analysis.json")
print("Done.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.


---
## Pipeline Summary

| Step | What | Status |
|------|------|--------|
| 1 | LLM Text Analysis (Gemini 2.5 Flash) | Done |
| 2 | Voice Mapping (edge-tts + Parler-TTS profiles) | Done |
| 3 | TTS Synthesis (edge-tts fast + Parler-TTS natural) | Done |
| 4 | SFX Generation (procedural ambient sounds) | Done |
| 5 | Audio Stitching (pydub, proper pacing) | Done |
| 6 | Human-in-the-Loop (Gradio editor) | Done |
| 7 | Evaluation (WER, MOS proxy, Emotion F1, RTF) | Done |

### Key Technologies Used
- **LLM:** Gemini 2.5 Flash (free API) for text analysis and segmentation
- **RAG-style Voice Selection:** Character attributes retrived from LLM analysis map to voice profiles
- **TTS (Option A):** edge-tts — Microsoft Neural TTS, fast, free, emotion via prosody
- **TTS (Option B):** Parler-TTS Expresso — HuggingFace, emotion-aware, natural language control
- **Evaluation:** OpenAI Whisper (WER), DistilRoBERTa (Emotion F1), automated MOS proxy

### For Your Report
- Run with 2-3 different text samples (novel excerpt, dialogue-heavy, narration-heavy)
- Do a small human MOS study (5-10 listeners, rate 1-5 on naturalness)
- Compare edge-tts vs Parler-TTS across all metrics
